# Data Cleaning for Network Traffic

Handle missing values, duplicates, and clean network traffic datasets using the SENTINEL src modules.

## 1. Create Synthetic Network Traffic Data

In [1]:
import pandas as pd
import numpy as np
import sys
sys.path.insert(0, '../')

from src import data_cleaning

np.random.seed(42)

# Generate network flow data with intentional missing values
data = {
    'Flow Duration': [500, 450, np.nan, 520, 100, 2000, 480, 510],
    'Total Packets': [10, 12, 8, np.nan, 1000, 50, 9, 11],
    'Bytes Sent': [2000, 2100, 1900, 2050, 50000, 3000, 2050, np.nan],
    'Destination Port': [443, 80, 22, 443, 3306, 8080, 443, 80],
    'Timestamp': pd.date_range('2024-01-01', periods=8, freq='1S')
}

df = pd.DataFrame(data)
print("Raw data with missing values:")
print(df)
print("\nData shape:", df.shape)

Raw data with missing values:
   Flow Duration  Total Packets  Bytes Sent  Destination Port  \
0          500.0           10.0      2000.0               443   
1          450.0           12.0      2100.0                80   
2            NaN            8.0      1900.0                22   
3          520.0            NaN      2050.0               443   
4          100.0         1000.0     50000.0              3306   
5         2000.0           50.0      3000.0              8080   
6          480.0            9.0      2050.0               443   
7          510.0           11.0         NaN                80   

            Timestamp  
0 2024-01-01 00:00:00  
1 2024-01-01 00:00:01  
2 2024-01-01 00:00:02  
3 2024-01-01 00:00:03  
4 2024-01-01 00:00:04  
5 2024-01-01 00:00:05  
6 2024-01-01 00:00:06  
7 2024-01-01 00:00:07  

Data shape: (8, 5)


C:\Users\ain kay\AppData\Local\Temp\ipykernel_18836\1325564607.py:16: FutureWarning: 'S' is deprecated and will be removed in a future version, please use 's' instead.
  'Timestamp': pd.date_range('2024-01-01', periods=8, freq='1S')


## 2. Detect Missing Values

In [2]:
# Use function from src.data_cleaning
missing_info = data_cleaning.detect_missing_values(df)

print("Missing Value Analysis:")
print(f"Columns with missing values: {missing_info['columns_with_missing']}")
print(f"\nMissing counts:")
for col, count in missing_info['missing_counts'].items():
    pct = missing_info['missing_percentages'][col]
    print(f"  {col}: {count} missing ({pct:.1f}%)")

Missing Value Analysis:
Columns with missing values: ['Flow Duration', 'Total Packets', 'Bytes Sent']

Missing counts:
  Flow Duration: 1 missing (12.5%)
  Total Packets: 1 missing (12.5%)
  Bytes Sent: 1 missing (12.5%)


## 3. Handle Missing Values - Median Strategy

In [3]:
# Fill missing values with median
numeric_cols = ['Flow Duration', 'Total Packets', 'Bytes Sent']
df_cleaned = data_cleaning.handle_missing_values(df, strategy='median', numeric_cols=numeric_cols)

print("Data after median imputation:")
print(df_cleaned)
print("\nMissing values remain:")
print(df_cleaned.isnull().sum())

Data after median imputation:
   Flow Duration  Total Packets  Bytes Sent  Destination Port  \
0          500.0           10.0      2000.0               443   
1          450.0           12.0      2100.0                80   
2          500.0            8.0      1900.0                22   
3          520.0           11.0      2050.0               443   
4          100.0         1000.0     50000.0              3306   
5         2000.0           50.0      3000.0              8080   
6          480.0            9.0      2050.0               443   
7          510.0           11.0      2050.0                80   

            Timestamp  
0 2024-01-01 00:00:00  
1 2024-01-01 00:00:01  
2 2024-01-01 00:00:02  
3 2024-01-01 00:00:03  
4 2024-01-01 00:00:04  
5 2024-01-01 00:00:05  
6 2024-01-01 00:00:06  
7 2024-01-01 00:00:07  

Missing values remain:
Flow Duration       0
Total Packets       0
Bytes Sent          0
Destination Port    0
Timestamp           0
dtype: int64


C:\Users\ain kay\Desktop\sentinel\sentinel\notebooks\..\src\data_cleaning.py:58: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_clean[col].fillna(df_clean[col].median(), inplace=True)


## 4. Create Data with Duplicates

In [4]:
# Create data with duplicates
data_dup = {
    'Flow Duration': [500, 450, 480, 500, 450, 520],
    'Total Packets': [10, 12, 8, 10, 12, 11],
    'Bytes Sent': [2000, 2100, 1900, 2000, 2100, 2050]
}

df_dup = pd.DataFrame(data_dup)
print("Data with duplicates:")
print(df_dup)
print(f"\nTotal rows: {len(df_dup)}")

Data with duplicates:
   Flow Duration  Total Packets  Bytes Sent
0            500             10        2000
1            450             12        2100
2            480              8        1900
3            500             10        2000
4            450             12        2100
5            520             11        2050

Total rows: 6


## 5. Detect and Remove Duplicates

In [5]:
# Use data_cleaning function to remove duplicates
df_unique = data_cleaning.remove_duplicates(df_dup)

print("Data after removing duplicates:")
print(df_unique)
print(f"\nTotal rows after: {len(df_unique)}")

Duplicates removed: 2
Data after removing duplicates:
   Flow Duration  Total Packets  Bytes Sent
0            500             10        2000
1            450             12        2100
2            480              8        1900
5            520             11        2050

Total rows after: 4


## 6. Extract Numeric Data and Handle Infinite Values

In [6]:
# Create data with infinite values
data_inf = {
    'Flow Duration': [500, 450, 480, np.inf, 520],
    'Total Packets': [10, 12, 8, 11, -np.inf],
    'Bytes Sent': [2000, 2100, 1900, 2050, 3000],
    'Port': [443, 80, 22, 443, 8080]
}

df_inf = pd.DataFrame(data_inf)
print("Data with infinite values:")
print(df_inf)

# Extract and clean numeric data
df_numeric = data_cleaning.extract_numeric_data(df_inf, handle_inf=True)
print("\nCleaned numeric data:")
print(df_numeric)

Data with infinite values:
   Flow Duration  Total Packets  Bytes Sent  Port
0          500.0           10.0        2000   443
1          450.0           12.0        2100    80
2          480.0            8.0        1900    22
3            inf           11.0        2050   443
4          520.0           -inf        3000  8080

Cleaned numeric data:
   Flow Duration  Total Packets  Bytes Sent  Port
0          500.0           10.0        2000   443
1          450.0           12.0        2100    80
2          480.0            8.0        1900    22
3          490.0           11.0        2050   443
4          520.0           10.5        3000  8080


## 7. Summary Statistics

In [7]:
# Generate summary statistics
summary = data_cleaning.get_summary_statistics(df_numeric)
print("Summary Statistics:")
print(summary)

# More detailed stats
print("\nCustom Statistics:")
for col in df_numeric.columns:
    data = df_numeric[col]
    print(f"\n{col}:")
    print(f"  Count: {data.count()}")
    print(f"  Mean: {data.mean():.2f}")
    print(f"  Median: {data.median():.2f}")
    print(f"  Std: {data.std():.2f}")
    print(f"  Min: {data.min():.2f}")
    print(f"  Max: {data.max():.2f}")

Summary Statistics:
       Flow Duration  Total Packets   Bytes Sent         Port
count       5.000000        5.00000     5.000000     5.000000
mean      488.000000       10.30000  2210.000000  1813.600000
std        25.884358        1.48324   447.772264  3508.562996
min       450.000000        8.00000  1900.000000    22.000000
25%       480.000000       10.00000  2000.000000    80.000000
50%       490.000000       10.50000  2050.000000   443.000000
75%       500.000000       11.00000  2100.000000   443.000000
max       520.000000       12.00000  3000.000000  8080.000000

Custom Statistics:

Flow Duration:
  Count: 5
  Mean: 488.00
  Median: 490.00
  Std: 25.88
  Min: 450.00
  Max: 520.00

Total Packets:
  Count: 5
  Mean: 10.30
  Median: 10.50
  Std: 1.48
  Min: 8.00
  Max: 12.00

Bytes Sent:
  Count: 5
  Mean: 2210.00
  Median: 2050.00
  Std: 447.77
  Min: 1900.00
  Max: 3000.00

Port:
  Count: 5
  Mean: 1813.60
  Median: 443.00
  Std: 3508.56
  Min: 22.00
  Max: 8080.00


## 8. Detect Outliers in Cleaned Data

In [ ]:
# Detect outliers using configured methods
duration = df_numeric['Flow Duration']

# IQR method
iqr_outliers = data_cleaning.detect_outliers_simple(duration, method='iqr', threshold=1.5)
print("IQR Outliers:")
print(f"  Detected: {iqr_outliers.sum()} outliers")
print(f"  Indices: {np.where(iqr_outliers)[0].tolist()}")
print(f"  Values: {duration[iqr_outliers].values}")

# Z-score method
zscore_outliers = data_cleaning.detect_outliers_simple(duration, method='zscore', threshold=2)
print(f"\nZ-Score Outliers:")
print(f"  Detected: {zscore_outliers.sum()} outliers")
print(f"  Indices: {np.where(zscore_outliers)[0].tolist()}")

## 9. Data Quality Report

In [ ]:
def generate_quality_report(df):
    """Generate a data quality report."""
    print("=" * 50)
    print("DATA QUALITY REPORT")
    print("=" * 50)
    print(f"\nShape: {df.shape[0]} rows × {df.shape[1]} columns")
    print(f"\nData Types:")
    print(df.dtypes)
    print(f"\nMissing Values:")
    missing = df.isnull().sum()
    if missing.sum() == 0:
        print("  None - Data is complete!")
    else:
        print(missing[missing > 0])
    print(f"\nDuplicates: {df.duplicated().sum()}")
    print(f"\nNumeric Columns: {df.select_dtypes(include='number').shape[1]}")
    print("\n" + "=" * 50)

generate_quality_report(df_numeric)

DATA QUALITY REPORT

Shape: 5 rows × 4 columns

Data Types:
Flow Duration    float64
Total Packets    float64
Bytes Sent         int64
Port               int64
dtype: object

Missing Values:
  None - Data is complete!

Duplicates: 0

Numeric Columns: 4



## Summary

- **Missing Values**: Detect with `.isnull()`, fill with median/mean
- **Duplicates**: Identify and remove with `.drop_duplicates()`
- **Infinite Values**: Replace before analysis
- **Quality**: Always inspect data before modeling
- **SENTINEL src**: Use reusable functions for consistent processing

Clean data = Better anomaly detection results!